# AG2 Multi-Agent RAG with ChromaDB

<a href="https://colab.research.google.com/github/chroma-core/chroma/blob/main/examples/use_with/ag2/ag2_multiagent_rag.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

This notebook demonstrates how to use [AG2](https://ag2.ai/) (formerly AutoGen)
multi-agent conversations with [ChromaDB](https://www.trychroma.com/) for
Retrieval-Augmented Generation (RAG).

**ChromaDB** is an open-source embedding database that handles tokenization,
embedding, and indexing automatically. It runs in-memory or with local persistence
— no server setup needed.

**AG2** (formerly AutoGen) is an open-source multi-agent conversation framework.

Two AG2 agents collaborate:
- **Research Agent**: Retrieves relevant documents from ChromaDB via semantic search
- **Analyst Agent**: Synthesizes retrieved information into comprehensive answers

ChromaDB auto-embeds documents using its built-in embedding function
(all-MiniLM-L6-v2) — no external embedding API key required.

In [ ]:
!pip install -q chromadb "ag2[openai]>=0.11.4,<1.0"

In [ ]:
import os

from autogen import (
    AssistantAgent,
    GroupChat,
    GroupChatManager,
    LLMConfig,
    UserProxyAgent,
)
import chromadb

# Set your OpenAI API key (needed only for LLM, NOT for embeddings)
os.environ["OPENAI_API_KEY"] = "your-api-key"  # Replace with your key

## Create ChromaDB Collection

ChromaDB runs in-memory with `EphemeralClient()`. Documents are automatically
embedded using the built-in sentence-transformers model — no embedding API
key needed.

In [ ]:
# Create in-memory ChromaDB client
client = chromadb.EphemeralClient()

# Create a collection (uses default all-MiniLM-L6-v2 embedding)
collection = client.create_collection("knowledge_base")

# Sample documents about AI topics
collection.add(
    documents=[
        "Retrieval-Augmented Generation (RAG) is a technique that combines "
        "information retrieval with language model generation. It first "
        "retrieves relevant documents from a knowledge base, then uses them "
        "as context for generating accurate, grounded responses. RAG reduces "
        "hallucination and enables models to access up-to-date information "
        "beyond their training data.",
        "Vector databases store data as high-dimensional vectors (embeddings) "
        "and enable fast similarity search. ChromaDB is an open-source "
        "embedding database that handles tokenization, embedding, and "
        "indexing automatically — making it easy to build semantic search "
        "applications.",
        "Multi-agent systems use multiple AI agents that collaborate to solve "
        "complex tasks. Each agent can have specialized roles, tools, and "
        "knowledge. AG2 (formerly AutoGen) is a popular framework for "
        "building multi-agent conversations where agents use tools and "
        "coordinate through structured dialogue patterns.",
        "Embedding models convert text into dense numerical vectors that "
        "capture semantic meaning. Similar texts produce similar vectors, "
        "enabling semantic search. ChromaDB uses all-MiniLM-L6-v2 by default "
        "and supports custom embedding functions for different models.",
        "Chunking is the process of splitting documents into smaller pieces "
        "for embedding and retrieval. Common strategies include fixed-size "
        "chunking, recursive character splitting, and semantic chunking. "
        "Optimal chunk size depends on the use case.",
    ],
    ids=[
        "doc_rag",
        "doc_vectordb",
        "doc_multiagent",
        "doc_embeddings",
        "doc_chunking",
    ],
    metadatas=[
        {"topic": "rag"},
        {"topic": "vector_databases"},
        {"topic": "agents"},
        {"topic": "embeddings"},
        {"topic": "chunking"},
    ],
)

print(f"Indexed {collection.count()} documents into ChromaDB")

## Test Semantic Search

Verify that ChromaDB retrieval works before connecting it to AG2 agents.

In [ ]:
# Semantic search — ChromaDB embeds the query automatically
results = collection.query(
    query_texts=["What is RAG?"],
    n_results=3,
)

for doc, distance in zip(
    results["documents"][0], results["distances"][0]
):
    print(f"Distance: {distance:.4f}")
    print(f"  {doc[:100]}...")
    print()

## Define ChromaDB Search Tool for AG2

Create a search function that AG2 agents will use as a registered tool
to retrieve relevant documents from the ChromaDB collection.

In [ ]:
# Configure AG2
llm_config = LLMConfig(
    {
        "model": "gpt-4o-mini",
        "api_key": os.getenv("OPENAI_API_KEY"),
        "api_type": "openai",
    }
)

# Create UserProxy (orchestrator)
user_proxy = UserProxyAgent(
    name="user_proxy",
    human_input_mode="NEVER",
    max_consecutive_auto_reply=10,
    code_execution_config=False,
)

# Research Agent — retrieves documents from ChromaDB
research_agent = AssistantAgent(
    name="research_agent",
    system_message=(
        "You are a Research Agent. Your role is to retrieve relevant "
        "documents from the ChromaDB collection using the search tool. "
        "Always use the search_documents tool to find information before "
        "answering. Present the retrieved documents clearly with their "
        "relevance scores."
    ),
    llm_config=llm_config,
)

# Analyst Agent — synthesizes information
analyst_agent = AssistantAgent(
    name="analyst_agent",
    system_message=(
        "You are an Analyst Agent. Your role is to synthesize information "
        "retrieved by the Research Agent into comprehensive, "
        "well-structured answers. Wait for the Research Agent to provide "
        "retrieved documents, then analyze and synthesize the findings. "
        "When done, say TERMINATE."
    ),
    llm_config=llm_config,
)


# Register search tool using AG2 decorator pattern
@user_proxy.register_for_execution()
@research_agent.register_for_llm(
    description="Search the ChromaDB collection for relevant documents"
)
def search_documents(query: str, num_results: int = 3) -> str:
    """Search ChromaDB for documents matching the query."""
    results = collection.query(
        query_texts=[query],
        n_results=num_results,
    )
    output = []
    for doc, dist in zip(
        results["documents"][0], results["distances"][0]
    ):
        output.append(f"[Distance: {dist:.4f}] {doc}")
    return (
        "\n\n".join(output)
        if output
        else "No relevant documents found."
    )

## Set Up Multi-Agent Conversation

Create a GroupChat where the Research Agent and Analyst Agent
collaborate to answer questions using ChromaDB-backed retrieval.

In [ ]:
# Create GroupChat with both agents
group_chat = GroupChat(
    agents=[user_proxy, research_agent, analyst_agent],
    messages=[],
    max_round=10,
)

manager = GroupChatManager(
    groupchat=group_chat, llm_config=llm_config
)

# Run multi-agent RAG conversation
result = user_proxy.run(
    manager,
    message=(
        "What is RAG and how do multi-agent systems enhance it? "
        "Search the knowledge base for relevant information."
    ),
)
result.process()

## Summary

This notebook demonstrated:

1. **ChromaDB as embedding database** — auto-embeds documents, no API key for embeddings
2. **AG2 multi-agent orchestration** — Research Agent retrieves, Analyst Agent synthesizes
3. **Tool registration** — AG2's decorator pattern wraps ChromaDB query as an agent tool
4. **GroupChat coordination** — agents collaborate through structured conversation

### Key Advantage
ChromaDB handles embedding automatically — you only need an OpenAI API key
for the LLM, not for document embedding or search.

### Learn More
- [AG2 Documentation](https://docs.ag2.ai/)
- [ChromaDB Documentation](https://docs.trychroma.com/)
- [AG2 GitHub](https://github.com/ag2ai/ag2)
- [ChromaDB GitHub](https://github.com/chroma-core/chroma)